# 📊 Tüm BIST (~606 Hisse) HMM+MOM3 Hibrit Tarama Sistemi
## Veri: TradingView (tvdatafeed) | Evren: Tüm BIST | Timeframe: Haftalık

**Kapsam:** TradingView scanner API → tüm BIST hisseleri (~600-650 sembol, dinamik)  
**Veri:** [rongardF/tvdatafeed](https://github.com/rongardF/tvdatafeed) → Haftalık OHLCV  
**Strateji:** GaussianHMM boğa/ayı rejimi × 13-haftalık momentum × MA trend filtresi  
**Projeksiyon:** Kalman Filtresi [seviye, hız] → 13 haftalık hedef fiyat  
**Pozisyon:** Quarter-Kelly + 2×ATR(14) stop-loss  

> **Çalışma Süresi:** ~15-25 dk (600 hisse download + HMM WFO)  
> **Tavsiye:** İlk çalıştırmada `CACHE_DATA=True` — veriyi diske kaydeder, sonraki çalıştırmalar hızlı olur.


In [ ]:
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

try:
    from tvDatafeed import TvDatafeed, Interval
except ImportError:
    print("tvDatafeed kuruluyor...")
    pip("git+https://github.com/rongardF/tvdatafeed.git")
    from tvDatafeed import TvDatafeed, Interval

try:
    from hmmlearn.hmm import GaussianHMM
except ImportError:
    print("hmmlearn kuruluyor...")
    pip("hmmlearn")
    from hmmlearn.hmm import GaussianHMM

try:
    from tqdm.notebook import tqdm
except ImportError:
    pip("tqdm")
    from tqdm.notebook import tqdm

print("✅ Tüm kütüphaneler hazır.")


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import requests, time, os, json, pickle
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.linalg import inv
from hmmlearn.hmm import GaussianHMM
from tvDatafeed import TvDatafeed, Interval
from tqdm.notebook import tqdm

pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
print("✅ Import'lar tamam.")


In [ ]:
# ═══════════════════════════════════════════════════════
#  AYARLAR — İhtiyacınıza göre değiştirin
# ═══════════════════════════════════════════════════════

INTERVAL    = Interval.in_weekly    # Haftalık (HMM için optimal)
N_BARS      = 200                   # 200 hafta ≈ 4 yıl lookback
EXCHANGE    = "BIST"
BENCHMARK   = "XU100"

# Önbellekleme — True ise veriyi diske kaydeder (sonraki çalıştırmalar hızlı)
CACHE_DATA   = True
CACHE_FILE   = "bist_weekly_cache.pkl"

# Filtreleme
MIN_BARS     = 52      # En az 52 hafta (1 yıl) veri olmalı
MIN_VOLUME   = 1000    # Haftalık ortalama hacim — çok düşük olanları atla
MAX_SYMBOLS  = 700     # TradingView'den alınacak maks sembol sayısı

# WFO parametreleri
TRAIN_MIN    = 52      # 1 yıl minimum eğitim
TEST_SIZE    = 13      # ~3 aylık test penceresi
STEP         = 13      # ~3 aylık adım

# TradingView hesabı (opsiyonel — anonim çalışır ama daha yavaş)
TV_USERNAME  = ""      # "email@example.com"
TV_PASSWORD  = ""      # "sifreniz"

print("✅ Konfigürasyon yüklendi.")
print(f"   Interval : {INTERVAL}")
print(f"   N_Bars   : {N_BARS}")
print(f"   Cache    : {CACHE_DATA}")
print(f"   Max sym  : {MAX_SYMBOLS}")


In [ ]:
if TV_USERNAME and TV_PASSWORD:
    tv = TvDatafeed(TV_USERNAME, TV_PASSWORD)
    print("✅ TradingView hesabıyla bağlandı.")
else:
    tv = TvDatafeed()
    print("✅ Anonim TvDatafeed bağlantısı kuruldu.")
    print("   (Hesapla bağlanmak için TV_USERNAME ve TV_PASSWORD doldurun)")

# Bağlantı testi
try:
    _t = tv.get_hist("THYAO", EXCHANGE, interval=INTERVAL, n_bars=5)
    print(f"\n✅ Bağlantı testi başarılı — THYAO {len(_t)} bar")
except Exception as e:
    print(f"\n❌ Bağlantı hatası: {e}")
    print("   VPN deneyin veya TradingView hesabıyla giriş yapın.")


In [ ]:
def get_all_bist_symbols(max_count=700):
    """
    TradingView Scanner API üzerinden tüm BIST hisselerini çeker.
    ~600-650 sembol döndürür (borsa kayıtlı tüm hisseler).
    """
    url = "https://scanner.tradingview.com/turkey/scan"
    payload = {
        "filter": [
            {"left": "type", "operation": "equal", "right": "stock"},
        ],
        "options": {"lang": "tr"},
        "symbols": {"query": {"types": ["stock"]}, "tickers": []},
        "columns": [
            "name", "close", "volume", "market_cap_basic",
            "average_volume_10d_calc", "exchange"
        ],
        "sort": {"sortBy": "market_cap_basic", "sortOrder": "desc"},
        "range": [0, max_count]
    }

    try:
        r = requests.post(url, json=payload, timeout=30,
                          headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        data = r.json()
        rows = data.get("data", [])
        
        symbols = []
        volumes = []
        for row in rows:
            sym_full = row.get("s", "")       # "BIST:THYAO"
            d        = row.get("d", [])
            
            # Sadece BIST hisseleri
            if not sym_full.startswith("BIST:"):
                continue
            ticker = sym_full.split(":")[1]
            
            # Hacim filtresi (index türevlerini ve çok düşük hacimli olanları atla)
            vol = d[2] if len(d) > 2 and d[2] else 0
            avg_vol = d[4] if len(d) > 4 and d[4] else 0
            
            symbols.append(ticker)
            volumes.append(avg_vol or vol)
        
        print(f"✅ TradingView Scanner: {len(symbols)} BIST sembolü bulundu")
        return symbols, volumes
        
    except Exception as e:
        print(f"❌ Scanner API hatası: {e}")
        print("   Yedek liste kullanılıyor...")
        return None, None

# Sembolleri çek
ALL_SYMBOLS, VOLUMES = get_all_bist_symbols(MAX_SYMBOLS)

if ALL_SYMBOLS is None:
    # Yedek: Manuel BIST listesi (kısmi)
    ALL_SYMBOLS = [
        "THYAO","GARAN","ASELS","BIMAS","EREGL","KCHOL","AKBNK","TUPRS",
        "FROTO","SISE","HALKB","VAKBN","MGROS","ASTOR","TKFEN","ISCTR",
        "TOASO","CCOLA","ENKAI","SAHOL","YKBNK","TCELL","PETKM","PGSUS",
        "KOZAL","OYAKC","ARCLK","KRDMD","DOHOL","SASA","TTKOM","AGHOL",
        "ULKER","AEFES","EKGYO","TAVHL","MAVI","BRSAN","GESAN","CWENE",
        "EUPWR","FENER","MIATK","PATEK","QUAGR","KTLEV","CVKMD","HEKTS",
        "PSGYO","ALARK","TABGD","SARKY","AKSEN","AKFGY","SOKM","ODAS",
        "KONTR","BERA","LOGO","OTKAR","CIMSA","VESTL","VESBE","VKGYO",
        "ZOREN","ENJSA","GLYHO","GUBRF","KOZAA","MPARK","NUHCM","ASUZU",
    ]
    VOLUMES = [0] * len(ALL_SYMBOLS)
    print(f"   Yedek liste: {len(ALL_SYMBOLS)} sembol")

print(f"\nTaranacak toplam hisse: {len(ALL_SYMBOLS)}")


In [ ]:
def safe_get(sym, exchange=EXCHANGE, interval=INTERVAL, n_bars=N_BARS, retries=3):
    for attempt in range(retries):
        try:
            df = tv.get_hist(symbol=sym, exchange=exchange,
                             interval=interval, n_bars=n_bars)
            if df is not None and len(df) >= MIN_BARS:
                df.columns = [c.lower() for c in df.columns]
                df.sort_index(inplace=True)
                df = df[~df.index.duplicated()]
                return df
            return None
        except Exception:
            if attempt < retries - 1:
                time.sleep(1.5 ** attempt)
    return None


# Önbellekten yükle (varsa)
RAW = {}
if CACHE_DATA and os.path.exists(CACHE_FILE):
    print(f"📂 Önbellekten yükleniyor: {CACHE_FILE}")
    with open(CACHE_FILE, "rb") as f:
        RAW = pickle.load(f)
    print(f"   {len(RAW)} hisse önbellekten yüklendi.")
    
    # Önbellekte olmayan yeni sembolleri bul
    missing = [s for s in ALL_SYMBOLS if s not in RAW]
    if missing:
        print(f"   {len(missing)} yeni sembol indirilecek...")
    else:
        print(f"   Tüm semboller önbellekte. İndirme atlanıyor.")
        ALL_SYMBOLS_TO_DL = []
else:
    missing = ALL_SYMBOLS

ALL_SYMBOLS_TO_DL = missing if CACHE_DATA and os.path.exists(CACHE_FILE) else ALL_SYMBOLS


# XU100 referans endeksini indir
print("\nXU100 benchmark indiriliyor...")
xu100_raw = safe_get(BENCHMARK, n_bars=N_BARS)
if xu100_raw is not None:
    XU100 = xu100_raw["close"].dropna()
    print(f"✅ XU100: {len(XU100)} bar ({XU100.index[0].date()} → {XU100.index[-1].date()})")
else:
    XU100 = None
    print("⚠️  XU100 indirilemedi — göreceli güç hesaplanamayacak.")


# Hisseleri İndir
if ALL_SYMBOLS_TO_DL:
    failed = []
    print(f"\n{len(ALL_SYMBOLS_TO_DL)} hisse indiriliyor...")
    print("(Her hisse arasında 0.25s bekleme — rate-limit önlemi)\n")
    
    for sym in tqdm(ALL_SYMBOLS_TO_DL, desc="İndiriliyor"):
        df = safe_get(sym)
        if df is not None:
            RAW[sym] = df
        else:
            failed.append(sym)
        time.sleep(0.25)
    
    if CACHE_DATA:
        with open(CACHE_FILE, "wb") as f:
            pickle.dump(RAW, f)
        print(f"\n💾 Önbellege kaydedildi: {CACHE_FILE}")
    
    print(f"\n✅ İndirme tamamlandı:")
    print(f"   Başarılı : {len(RAW)} hisse")
    print(f"   Başarısız: {len(failed)} hisse")
    if failed:
        print(f"   Başarısız semboller: {failed[:20]}{'...' if len(failed)>20 else ''}")
else:
    failed = []
    print(f"✅ Tüm {len(RAW)} hisse önbellekten hazır.")


In [ ]:
SPLIT_THRESH = -0.45  # Haftalık -%45'ten büyük düşüş → split (NaN yap)

def prepare(df_raw, xu100_series=None):
    df = df_raw[["open","high","low","close","volume"]].copy()
    df.dropna(subset=["close"], inplace=True)
    df["close"] = pd.to_numeric(df["close"], errors="coerce")
    df.dropna(subset=["close"], inplace=True)
    if len(df) < MIN_BARS:
        return None

    # Log getiri + split tespiti
    df["log_ret"] = np.log(df["close"] / df["close"].shift(1))
    df["is_split"] = df["log_ret"] < SPLIT_THRESH
    df.loc[df["is_split"], "log_ret"] = np.nan

    # Momentum
    df["mom3w"]  = df["close"].pct_change(3)
    df["mom13w"] = df["close"].pct_change(13)

    # Göreceli güç vs XU100
    if xu100_series is not None:
        xu = xu100_series.reindex(df.index, method="ffill")
        xu_lr = np.log(xu / xu.shift(1))
        df["rel_xu100"] = df["log_ret"] - xu_lr
    else:
        df["rel_xu100"] = df["log_ret"] - df["log_ret"].rolling(26).mean()

    # SMA
    df["sma13"] = df["close"].rolling(13).mean()
    df["sma26"] = df["close"].rolling(26).mean()

    # RSI(14)
    d = df["close"].diff()
    g = d.clip(lower=0).rolling(14).mean()
    l = (-d.clip(upper=0)).rolling(14).mean()
    df["rsi14"] = 100 - 100/(1 + g/l.clip(lower=1e-9))

    # ATR(14)
    tr = pd.concat([
        df["high"] - df["low"],
        (df["high"] - df["close"].shift(1)).abs(),
        (df["low"]  - df["close"].shift(1)).abs(),
    ], axis=1).max(axis=1)
    df["atr14"] = tr.rolling(14).mean()

    # Hacim oranı (son bar / 20 haftalık ortalama)
    df["vol_ratio"] = df["volume"] / df["volume"].rolling(20).mean().clip(lower=1)

    return df

# Tüm hisseler için indikatörleri hesapla
WEEKLY = {}
skipped = 0
for ticker, df_raw in RAW.items():
    try:
        df = prepare(df_raw, XU100)
        if df is not None and len(df) >= MIN_BARS:
            WEEKLY[ticker] = df
        else:
            skipped += 1
    except Exception:
        skipped += 1

print(f"✅ İndikatörler hesaplandı: {len(WEEKLY)} hisse (atlandı: {skipped})")


In [ ]:
# ─── HMM Stratejisi ────────────────────────────────────────────────────────
class HMMStrategy:
    """2-durumlu GaussianHMM — özellikler: [log_ret, rel_xu100]"""
    def __init__(self, n_iter=300, min_samples=30):
        self.n_iter, self.min_samples = n_iter, min_samples
        self.model, self.bull_state = None, 0

    def _X(self, df):
        return df[["log_ret","rel_xu100"]].replace([np.inf,-np.inf], np.nan).dropna()

    def fit(self, df):
        X = self._X(df)
        if len(X) < self.min_samples:
            return False
        try:
            m = GaussianHMM(n_components=2, covariance_type="full",
                            n_iter=self.n_iter, random_state=42)
            m.fit(X.values)
            self.model = m
            self.bull_state = int(np.argmax(m.means_[:, 0]))
            return True
        except:
            return False

    def predict(self, df):
        if not self.model: return pd.Series(0, index=df.index)
        X = self._X(df)
        if X.empty: return pd.Series(0, index=df.index)
        try:
            s = self.model.predict(X.values)
            return pd.Series((s == self.bull_state).astype(float),
                             index=X.index).reindex(df.index, fill_value=0)
        except: return pd.Series(0, index=df.index)

    def bull_prob(self, df):
        if not self.model: return 0.5
        X = self._X(df)
        if X.empty: return 0.5
        try:
            return float(self.model.predict_proba(X.values)[-1, self.bull_state])
        except: return 0.5


# ─── Walk-Forward Optimizasyon ──────────────────────────────────────────────
def wfo_hmm(df, train_min=TRAIN_MIN, test_size=TEST_SIZE, step=STEP):
    n = len(df)
    oos_r, oos_d = [], []
    for fs in range(train_min, n - test_size + 1, step):
        h = HMMStrategy()
        if not h.fit(df.iloc[:fs]): continue
        sig = h.predict(df.iloc[:fs+test_size])
        sig_t = sig.iloc[fs:fs+test_size].shift(1).fillna(0)
        ret_t = df["log_ret"].iloc[fs:fs+test_size].fillna(0)
        oos_r.extend((sig_t * ret_t).tolist())
        oos_d.extend(ret_t.index.tolist())
    if not oos_r:
        return {"sharpe": -99, "ret": -99, "n_folds": 0}
    s = pd.Series(oos_r, index=oos_d).sort_index()
    return {
        "sharpe":  round(float(s.mean()/(s.std()+1e-9)) * np.sqrt(52), 3),
        "ret":     round(float(np.exp(s.sum())-1)*100, 1),
        "n_folds": (n - train_min - test_size)//step + 1
    }


# ─── Kalman Projeksiyon ─────────────────────────────────────────────────────
def kalman_proj(close_s, n_fw=13):
    y = np.log(close_s.dropna().values)
    if len(y) < 20:
        p = float(close_s.iloc[-1])
        return 0, p, p, p
    F = np.array([[1,1],[0,1]]); H = np.array([[1,0]])
    Q = np.eye(2)*1e-4; R = np.array([[1e-2]])
    x = np.array([[y[0]],[0.]]); P = np.eye(2)
    for obs in y:
        xp = F@x; Pp = F@P@F.T+Q
        inn = obs-(H@xp)[0,0]; S = H@Pp@H.T+R; K = Pp@H.T@inv(S)
        x = xp+K*inn; P = (np.eye(2)-K@H)@Pp
    vel=float(x[1,0]); lev=float(x[0,0])
    fl = lev+vel*n_fw; sig = float(np.std(np.diff(y))*np.sqrt(n_fw))
    return (
        round(vel*52*100, 2),
        round(float(np.exp(fl)), 2),
        round(float(np.exp(fl-2*sig)), 2),
        round(float(np.exp(fl+2*sig)), 2),
    )

print("✅ HMM | WFO | Kalman modelleri hazır.")


In [ ]:
def scan_ticker(ticker):
    df = WEEKLY.get(ticker)
    if df is None or len(df) < MIN_BARS:
        return None
    
    df = df.dropna(subset=["log_ret","rel_xu100"])
    if len(df) < MIN_BARS:
        return None

    cur  = float(df["close"].iloc[-1])
    atr  = float(df["atr14"].iloc[-1])  if pd.notna(df["atr14"].iloc[-1])  else cur*0.03
    rsi  = float(df["rsi14"].iloc[-1])  if pd.notna(df["rsi14"].iloc[-1])  else 50
    vol  = float(df["vol_ratio"].iloc[-1]) if pd.notna(df["vol_ratio"].iloc[-1]) else 1.
    mom  = float(df["mom13w"].iloc[-1]) if pd.notna(df["mom13w"].iloc[-1]) else 0
    mom3 = float(df["mom3w"].iloc[-1])  if pd.notna(df["mom3w"].iloc[-1])  else 0
    sma13= float(df["sma13"].iloc[-1])  if pd.notna(df["sma13"].iloc[-1])  else cur
    sma26= float(df["sma26"].iloc[-1])  if pd.notna(df["sma26"].iloc[-1])  else cur

    wfo  = wfo_hmm(df)

    # Güncel HMM durumu
    hmm = HMMStrategy()
    hmm.fit(df)
    prob      = hmm.bull_prob(df)
    hmm_sig   = prob > 0.55
    mom_sig   = mom > 0
    ma_bull   = cur > sma13 > sma26

    # Kalman
    vel_pct, tgt, tgt_lo, tgt_hi = kalman_proj(df["close"])
    upside = (tgt/cur - 1)*100

    # Kompozit Skor: HMM %40 + Kalman hız %25 + MOM %20 + MA %10 + Hacim %5
    s_hmm  = prob * 100
    s_kal  = min(max(vel_pct, 0), 100)
    s_mom  = min(max(mom*200, 0), 100)
    s_ma   = 100 if ma_bull else 0
    s_vol  = min(vol*50, 100)
    comp   = 0.40*s_hmm + 0.25*s_kal + 0.20*s_mom + 0.10*s_ma + 0.05*s_vol

    # Sinyal
    if   hmm_sig and mom_sig and ma_bull: sinyal = "GÜÇLÜ AL"
    elif hmm_sig and mom_sig:             sinyal = "AL"
    elif hmm_sig:                         sinyal = "HMM AL"
    elif mom_sig and ma_bull:             sinyal = "MOM AL"
    else:                                 sinyal = "BEKLE"

    return dict(
        ticker=ticker, fiyat=cur, sinyal=sinyal,
        hmm_prob=round(prob*100,1), wfo_sharpe=wfo["sharpe"],
        wfo_ret=wfo["ret"], n_folds=wfo["n_folds"],
        mom13w=round(mom*100,1), mom3w=round(mom3*100,1),
        kal_vel=vel_pct, kal_hedef=tgt,
        kal_lo=tgt_lo, kal_hi=tgt_hi,
        kal_upside=round(upside,1),
        rsi=round(rsi,1), vol_ratio=round(vol,2),
        stop=round(cur - 2*atr, 2),
        atr=round(atr, 2),
        composite=round(comp,1),
    )

print("✅ Tarama fonksiyonu hazır.")


In [ ]:
print(f"🔍 Tüm BIST taraması başlıyor: {len(WEEKLY)} hisse")
print("   (HMM WFO hesabı — her hisse ~0.5-2 sn)\n")

results = []
errors  = []

for ticker in tqdm(list(WEEKLY.keys()), desc="HMM+WFO Tarama"):
    try:
        r = scan_ticker(ticker)
        if r:
            results.append(r)
    except Exception as e:
        errors.append((ticker, str(e)))

print(f"\n✅ Tarama tamamlandı:")
print(f"   Analiz edilen : {len(results)} hisse")
print(f"   Hata          : {len(errors)} hisse")

DF = (pd.DataFrame(results)
        .sort_values("composite", ascending=False)
        .reset_index(drop=True))
DF.index += 1

# Sinyal özeti
for sinyal in ["GÜÇLÜ AL","AL","HMM AL","MOM AL","BEKLE"]:
    count = len(DF[DF["sinyal"]==sinyal])
    emoji = {"GÜÇLÜ AL":"🟢","AL":"🟡","HMM AL":"🔵","MOM AL":"🟣","BEKLE":"🔴"}.get(sinyal,"")
    print(f"   {emoji} {sinyal:<10}: {count} hisse")


In [ ]:
COLS = ["ticker","fiyat","sinyal","hmm_prob","mom13w",
        "kal_hedef","kal_upside","kal_vel","wfo_sharpe","rsi","vol_ratio","stop","composite"]

date_str = pd.Timestamp.today().strftime("%Y-%m-%d")

print("=" * 110)
print(f"📊 TÜM BIST HMM+MOM3 HİBRİT TARAMA — {date_str}")
print(f"   {len(DF)} hisse analiz edildi | Timeframe: Haftalık | Lookback: {N_BARS} bar")
print("=" * 110)

# ── GÜÇLÜ AL ───────────────────────────────────────────────────────────────
guclu = DF[DF["sinyal"]=="GÜÇLÜ AL"]
print(f"\n🟢 GÜÇLÜ AL ({len(guclu)} hisse) — HMM boğa + MOM13W+ + SMA trend")
print("-"*110)
if len(guclu) > 0:
    print(guclu[COLS].to_string())
else:
    print("   Şu an bu koşulları karşılayan hisse yok.")

# ── AL ─────────────────────────────────────────────────────────────────────
al = DF[DF["sinyal"]=="AL"]
print(f"\n🟡 AL ({len(al)} hisse) — HMM boğa + MOM13W+ (SMA filtresi yok)")
print("-"*110)
if len(al) > 0:
    print(al.head(20)[COLS].to_string())
    if len(al) > 20:
        print(f"   ... ve {len(al)-20} hisse daha. DF[DF['sinyal']=='AL'] ile tamamını görün.")

# ── HMM AL ─────────────────────────────────────────────────────────────────
hmmal = DF[DF["sinyal"]=="HMM AL"]
print(f"\n🔵 HMM AL ({len(hmmal)} hisse) — Sadece HMM boğa rejimine girmiş")
print("-"*110)
if len(hmmal) > 0:
    print(hmmal.head(15)[COLS].to_string())

# ── Genel Sıralama (İlk 50) ────────────────────────────────────────────────
print(f"\n{'='*110}")
print(f"📈 GENEL SIRALAMA — İlk 50 (Kompozit Skor)")
print("-"*110)
print(DF.head(50)[COLS].to_string())

# CSV'ye kaydet
DF.to_csv(f"bist_tarama_{date_str}.csv", index=True)
print(f"\n💾 Sonuçlar kaydedildi: bist_tarama_{date_str}.csv")


In [ ]:
# En iyi 20 hisse için görsel (Güçlü AL + AL önce)
buy_first = pd.concat([
    DF[DF["sinyal"]=="GÜÇLÜ AL"],
    DF[DF["sinyal"]=="AL"],
    DF[DF["sinyal"]=="HMM AL"],
]).head(20)

if len(buy_first) == 0:
    buy_first = DF.head(20)

n = len(buy_first)
ncols = 5
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(24, nrows*4))
axes = axes.flatten() if nrows > 1 else axes.reshape(1,-1).flatten()

fig.suptitle(
    f"Tüm BIST — HMM+MOM3 En İyi {n} Sinyal | {date_str}\n"
    "Haftalık Fiyat + HMM Rejimleri (Yeşil=Boğa, Kırmızı=Ayı) + Kalman Hedef (--) + Stop (:)",
    fontsize=12, fontweight="bold"
)

hmm_cache = {}

for idx, (_, row) in enumerate(buy_first.iterrows()):
    ax  = axes[idx]
    tkr = row["ticker"]
    
    if tkr not in WEEKLY:
        ax.set_visible(False)
        continue

    df  = WEEKLY[tkr].tail(104)   # Son 2 yıl

    if tkr not in hmm_cache:
        h = HMMStrategy()
        h.fit(WEEKLY[tkr])
        hmm_cache[tkr] = h
    states = hmm_cache[tkr].predict(WEEKLY[tkr]).reindex(df.index).fillna(0)

    ax.plot(df.index, df["close"], color="black", lw=1.3, zorder=5)

    # Rejim arka planı
    for j in range(len(df)-1):
        c = "#d0f5d0" if states.iloc[j]==1 else "#f5d0d0"
        ax.axvspan(df.index[j], df.index[j+1], alpha=0.3, color=c, zorder=1)

    ax.plot(df.index, df["sma13"], color="darkorange", lw=0.8, alpha=0.8)
    ax.plot(df.index, df["sma26"], color="purple",     lw=0.8, alpha=0.8)
    ax.axhline(row["kal_hedef"], color="blue",   ls="--", lw=0.9, alpha=0.8)
    ax.axhline(row["stop"],      color="crimson", ls=":",  lw=0.8, alpha=0.8)

    # Kalman bant
    ax.axhline(row["kal_lo"], color="blue", ls="-", lw=0.4, alpha=0.3)
    ax.axhline(row["kal_hi"], color="blue", ls="-", lw=0.4, alpha=0.3)

    tc = {"GÜÇLÜ AL":"darkgreen","AL":"darkorange","HMM AL":"steelblue",
          "MOM AL":"purple"}.get(row["sinyal"], "gray")
    ax.set_title(
        f"{tkr} [{row['sinyal']}]\n"
        f"Skor:{row['composite']:.0f}  HMM:{row['hmm_prob']:.0f}%  "
        f"Hedef:{row['kal_hedef']:.0f}({row['kal_upside']:+.0f}%)",
        fontsize=8, color=tc, fontweight="bold"
    )
    ax.tick_params(labelsize=6)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

for i in range(idx+1, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
out_img = f"bist_hmm_{date_str}.png"
plt.savefig(out_img, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Grafik kaydedildi: {out_img}")


In [ ]:
print("\n" + "═"*80)
print("💼 AL SİNYALLERİ — QUARTER-KELLY POZİSYON ÖNERİLERİ")
print("═"*80)

al_df = DF[DF["sinyal"].isin(["GÜÇLÜ AL","AL"])].copy()

if len(al_df) == 0:
    print("Şu an Güçlü AL veya AL sinyali yok.")
    print("HMM AL sinyallerini incelemeyi düşünün:")
    al_df = DF[DF["sinyal"]=="HMM AL"].head(5)

pos_list = []
for _, r in al_df.head(15).iterrows():
    p    = r["hmm_prob"]/100
    b    = max(r["kal_upside"]/100, 0.05)
    risk = max((r["fiyat"]-r["stop"])/r["fiyat"], 0.01)
    kelly = max((p*b-(1-p)*risk)/b, 0)
    qk    = min(kelly*0.25, 0.15)
    rr    = b/risk if risk>0 else 0
    pos_list.append({
        "Hisse":       r["ticker"],
        "Sinyal":      r["sinyal"],
        "Fiyat":       r["fiyat"],
        "Hedef(13H)":  r["kal_hedef"],
        "Stop(2xATR)": r["stop"],
        "Risk%":       round(risk*100,1),
        "Ödül%":       round(b*100,1),
        "R:R":         round(rr,2),
        "Pozisyon%":   round(qk*100,1),
        "HMM%":        r["hmm_prob"],
        "WFOSharpe":   r["wfo_sharpe"],
        "Skor":        r["composite"],
    })

if pos_list:
    pos_df = pd.DataFrame(pos_list).sort_values("Pozisyon%", ascending=False)
    pos_df.reset_index(drop=True, inplace=True); pos_df.index+=1
    total = pos_df["Pozisyon%"].sum()
    print(pos_df.to_string())
    print("-"*80)
    print(f"Toplam yatırım : %{total:.1f}")
    print(f"Nakit / tahvil : %{100-total:.1f}")
    print("\n⚠️  Bu çıktı yatırım tavsiyesi değildir.")

# Tüm sonuçları CSV kaydet
DF.to_csv(f"bist_tarama_tam_{pd.Timestamp.today().strftime('%Y-%m-%d')}.csv")
print(f"\n💾 Tam sonuçlar CSV olarak kaydedildi.")
